# L49 · 去相关 — DCT 离散余弦变换

**目标**：从零实现 DCT-II，理解它如何把相关的 Mel 特征向量压缩成近似独立的倒谱系数。

🔗 Aurora 连接：`aurora.audio.mfcc.dct_ii()` 是本节的标准答案，最终 MFCC 流水线直接调用它。

Mel 滤波器组输出的是一组对数能量值，相邻 bin 因滤波器重叠而高度相关。
DCT-II 用一组余弦基把这些相关分量"旋转"到新坐标系，使得绝大多数信息集中在前几个系数（低阶倒谱），
高阶系数近似独立且幅度趋零——这正是 MFCC 的核心一步。

In [ ]:
import numpy as np

## 1. DCT-II 公式

DCT-II 把长度为 N 的实值序列 x 变换为同样长度的系数序列 X：

```
X[k] = sum( x[n] * cos(pi * (2*n + 1) * k / (2*N)), n=0..N-1 ),  k = 0..N-1
```

每个输出 X[k] 是 x 与第 k 个余弦基向量的内积。k=0 时余弦为常数 1，X[0] 是 x 的（未归一化）均值；k 越大，余弦振荡越快。

In [ ]:
# 演示：观察 DCT-II 的余弦基向量（N=8）
N = 8
k_vals = np.arange(N)
n_vals = np.arange(N)
# M[k, n] = cos(pi*(2n+1)*k / (2N))
M = np.cos(np.pi * np.outer(k_vals, 2*n_vals + 1) / (2*N))
print("余弦矩阵 M，shape:", M.shape)
print("第 0 行（k=0，常数基）:", np.round(M[0], 4))
print("第 1 行（k=1，最低频余弦）:", np.round(M[1], 4))
print("第 4 行（k=4，较高频余弦）:", np.round(M[4], 4))

## 2. DCT vs DFT：实值信号的更高效表示

DFT 假设信号以 N 为周期循环，当信号两端不连续时会产生"吉布斯效应"（频谱泄漏）。
DCT-II 等效于将信号做偶对称延拓后取 DFT 的实部，天然没有泄漏，输出为纯实数。

正交性验证：`M @ M.T / N` 应等于 2 倍单位阵（除第 0 行/列外），说明余弦基向量两两正交。
正因如此，DCT 矩阵在适当归一化后是正交矩阵，变换可精确逆转。

In [ ]:
# 演示：验证余弦矩阵的正交性
N = 8
k_vals = np.arange(N)
n_vals = np.arange(N)
M = np.cos(np.pi * np.outer(k_vals, 2*n_vals + 1) / (2*N))
G = M @ M.T  # Gram 矩阵，理论上对角线为 N/2（k>0）或 N（k=0）
print("M @ M.T 对角线（应为 N/2=4 或 N=8）:")
print(np.round(np.diag(G), 6))
print("非对角线最大绝对值（应≈0）:", np.max(np.abs(G - np.diag(np.diag(G)))))

## 3. 去相关性：从 Mel 能量到倒谱系数

Mel 滤波器组相邻通道重叠约 50%，导致相邻能量值的 Pearson 相关系数可达 0.9 以上。
DCT-II 将这批相关特征投影到正交余弦基上：低阶系数（k=0..12）捕获说话人/音素的整体频谱形状，
高阶系数（k≥13）捕获细节且相关性极低——这使得以欧氏距离为核心的分类器（GMM、DTW、神经网络）更加高效。
通常只取前 13 个系数，即 MFCC-13。

In [ ]:
# 演示：模拟相关的 Mel 能量，经 DCT 后的系数分布
rng = np.random.default_rng(42)
# 用 AR(1) 过程模拟相邻 Mel bin 高度相关（phi=0.9）
N = 26
mel_energy = np.zeros(N)
mel_energy[0] = rng.standard_normal()
for i in range(1, N):
    mel_energy[i] = 0.9 * mel_energy[i-1] + 0.1 * rng.standard_normal()

# 手算 DCT
k = np.arange(N); n = np.arange(N)
M = np.cos(np.pi * np.outer(k, 2*n + 1) / (2*N))
dct_out = M @ mel_energy

print("Mel 能量（前5）:", np.round(mel_energy[:5], 4))
print("DCT 系数（全部，绝对值）:", np.round(np.abs(dct_out), 4))
print("能量集中度：前13系数占总能量的比例:",
      np.round(np.sum(dct_out[:13]**2) / np.sum(dct_out**2), 4))

## 4. ✏️ 实现 `dct_ii(x)`

**签名**：`x: np.ndarray (1D float) -> np.ndarray (1D float)`，输出 shape 与输入相同。

**推理路线**：
1. `N = len(x)`；分别构建 `k = np.arange(N)`（行索引）和 `n = np.arange(N)`（列索引）
2. 用 `np.outer(k, 2*n + 1)` 计算指数矩阵，乘以 `pi / (2*N)` 后取余弦，得到余弦矩阵 `M`，shape `(N, N)`
3. `return M @ x`——矩阵向量乘法，每行点积即对应 X[k]

**参考输入输出**：

```
x = np.array([1., 2., 3., 4.])
dct_ii(x) ≈ [10.0, -6.3066, 0.0, -0.4449]
# 与 scipy.fft.dct([1,2,3,4], type=2, norm=None) 完全一致（atol=1e-8）
```

<details><summary>点击查看参考实现</summary>

```python
def dct_ii(x: np.ndarray) -> np.ndarray:
    N = len(x)
    k = np.arange(N)
    n = np.arange(N)
    M = np.cos(np.pi * np.outer(k, 2 * n + 1) / (2 * N))
    return M @ x
```

</details>

In [ ]:
def dct_ii(x: np.ndarray) -> np.ndarray:
    """DCT-II（无归一化）：X[k] = sum(x[n]*cos(pi*(2n+1)*k/(2N)), n=0..N-1)"""
    # ✏️ TODO: 计算 N, k, n
    # ✏️ TODO: 构建余弦矩阵 M，shape (N, N)
    # ✏️ TODO: return M @ x
    pass

In [ ]:
# numpy-based DCT-II reference (no scipy needed)
def _dct_ii_ref(x):
    N = len(x)
    k = np.arange(N)
    n = np.arange(N)
    return 2 * (np.cos(np.pi * k[:, None] * (2*n[None, :] + 1) / (2*N)) @ x)

x = np.array([1., 2., 3., 4.])
ref = _dct_ii_ref(x)
out = dct_ii(x)
assert out is not None, "dct_ii 还没实现，请完成 TODO"
assert out.shape == x.shape, f"输出 shape 错误：{out.shape}"
assert np.allclose(out, ref, atol=1e-8), f"数值不匹配\n实现：{out}\n参考：{ref}"
print("✅ dct_ii([1,2,3,4]) 通过，输出:", np.round(out, 4))

# 更大规模测试
rng = np.random.default_rng(0)
for n_test in [8, 13, 26, 40]:
    xr = rng.standard_normal(n_test)
    assert np.allclose(dct_ii(xr), _dct_ii_ref(xr), atol=1e-8)
print("✅ 随机向量（N=8,13,26,40）全部通过")

## 5. 参数实验：可逆性与截断重建

**实验 A — 可逆性验证**：DCT-II 的逆变换是它自身的转置除以 `2N`（无归一化版本）：

```
x_rec = M.T @ X / (2*N)  # 其中第 0 行需额外除以 2（因 k=0 基的模是 k>0 的 2 倍）
```

实际上更简单的方式：用 `scipy.fft.idct(X, type=2, norm=None)` 验证精确重建。

**实验 B — 截断重建**：只保留前 `k_keep` 个 DCT 系数，其余置零后逆变换，观察重建误差随 k_keep 变化。
预期现象：k_keep=3 时误差很大；k_keep=6 大致捕获整体形状；k_keep=13 误差接近 0（典型 MFCC 截断点）。

In [ ]:
# numpy-based inverse DCT-II reference
def _idct_ii_ref(X):
    N = len(X)
    k = np.arange(N)
    n = np.arange(N)
    # IDCT-II: x[n] = (X[0] + 2*sum_{k=1}^{N-1} X[k]*cos(pi*k*(2n+1)/(2N))) / (2N)
    W = np.cos(np.pi * k[None, :] * (2*n[:, None] + 1) / (2*N))
    W[:, 0] /= 2  # k=0 term: X[0]/(2N) vs X[k!=0]*2/(2N)
    return W @ X / N

# 用一段 26 维模拟 Mel 能量做实验
rng = np.random.default_rng(7)
x_mel = rng.standard_normal(26)
X_dct = dct_ii(x_mel)

# 实验 A：精确逆变换
x_rec_full = _idct_ii_ref(X_dct)
print("实验 A — 精确重建误差（应≈0）:", np.max(np.abs(x_rec_full - x_mel)))

# 实验 B：截断重建
print("\n实验 B — 截断重建误差：")
for k_keep in [3, 6, 13]:
    X_trunc = X_dct.copy()
    X_trunc[k_keep:] = 0.0
    x_trunc_rec = _idct_ii_ref(X_trunc)
    rmse = np.sqrt(np.mean((x_trunc_rec - x_mel) ** 2))
    print(f"  k_keep={k_keep:2d} → RMSE={rmse:.4f}")

## 收束

`dct_ii(x)` 通过构建 N×N 余弦矩阵并做矩阵乘法，输出与输入等长的实值倒谱系数向量。
该函数是 `aurora.audio.mfcc` 模块中 MFCC 流水线的第三步，承接对数 Mel 能量谱，输出低阶系数即为 MFCC。
截断实验表明前 13 个系数已能精确重建信号，这解释了为何 MFCC-13 成为语音识别的标准特征维度。
下一节（Day 2）将把 `dct_ii` 嵌入完整的 MFCC 流水线：STFT → Mel 滤波 → 对数 → DCT → 截断，端到端处理真实音频帧。